# Fine-tuning based on Qwen3-1.7B with Style Vector Input

In [1]:
# Run this on Remote Jupyter Book.

import os

os.getcwd()
os.chdir("/root/OtakuLab")

## Build Dataset

### Define Training Set Format

In [1]:
from typing import Optional
from pathlib import Path
from typing_extensions import TypedDict
from difflib import SequenceMatcher
from sentence_transformers import SentenceTransformer, util
from tqdm import tqdm
import json

BGE_MODEL_PATH = "../Models/bge-large-zh-v1.5"
SEMANTIC_SIMILARITY_THRESHOLD = 0.96

DENSE_METRIC_KEYS = [
    "mdd",
    "verb_noun_ratio",
    "question_ratio",
    "imperative_ratio",
    "fragment_ratio",
    "marker_density",
]

class InstructionComponents(TypedDict):
    semantic_tags: list[str]
    structural_metrics: dict[str, float]
    signature_rules: list[str]

class DatasetItem(TypedDict):
    character: str
    neutral_sentence: str
    instruction_components: InstructionComponents
    thinking_process: str
    output: str

class DatasetStorage:
    def __init__(self) -> None:
        self.items: dict[str, DatasetItem] = {}
        self._semantic_model: Optional[SentenceTransformer] = None

    def __len__(self) -> int:
        return len(self.items)

    def _get_semantic_model(self) -> SentenceTransformer:
        if self._semantic_model is None:
            self._semantic_model = SentenceTransformer(BGE_MODEL_PATH)
        return self._semantic_model

    @staticmethod
    def _normalize_for_similarity(text: str) -> str:
        return "".join(ch.lower() for ch in text if ch.isalnum())

    @classmethod
    def _is_extremely_similar(cls, neutral_sentence: str, output: str) -> bool:
        neutral_norm = cls._normalize_for_similarity(neutral_sentence)
        output_norm = cls._normalize_for_similarity(output)

        if not neutral_norm or not output_norm:
            return False
        if neutral_norm == output_norm:
            return True

        matcher = SequenceMatcher(None, neutral_norm, output_norm)
        ratio = matcher.ratio()
        if ratio >= 0.93:
            return True

        shorter = min(len(neutral_norm), len(output_norm))
        longer = max(len(neutral_norm), len(output_norm))
        length_ratio = shorter / longer
        if length_ratio >= 0.86 and (neutral_norm in output_norm or output_norm in neutral_norm):
            return True

        longest_match = matcher.find_longest_match(0, len(neutral_norm), 0, len(output_norm)).size
        return ratio >= 0.88 and (longest_match / shorter) >= 0.93

    def _is_semantically_similar(self, neutral_sentence: str, output: str) -> bool:
        if not neutral_sentence.strip() or not output.strip():
            return False

        model = self._get_semantic_model()
        embeddings = model.encode(
            [neutral_sentence, output],
            normalize_embeddings=True,
            convert_to_numpy=True,
        )
        similarity = util.cos_sim(embeddings[0], embeddings[1]).item()

        return similarity >= SEMANTIC_SIMILARITY_THRESHOLD
    
    def calc_avg_similarity(self) -> float:
        if len(self.items) < 2:
            return 0.0

        total_similarity = 0.0
        count = 0

        outputs = list(self.items.keys())
        for i in tqdm(range(len(outputs)), desc="Calculating average similarity"):
            netural = self.items[outputs[i]]["neutral_sentence"]
            styled_sentence = self.items[outputs[i]]["output"]
            sim = SequenceMatcher(None, netural, styled_sentence).ratio()
            total_similarity += sim
            count += 1

        return total_similarity / count if count > 0 else 0.0

    def new_item(self, character: str, neutral_sentence: str, thinking_process: str, output: str) -> bool:
        """
        向数据存储中添加一个新的训练集条目

        :return: 如果条目成功添加返回 True, 如果 output 已存在或与 neutral_sentence 极其相似则不添加并返回 False
        """
        if output in self.items:
            return False

        if self._is_extremely_similar(neutral_sentence, output):
            return False

        if self._is_semantically_similar(neutral_sentence, output):
            return False

        item: DatasetItem = {
            "character": character,
            "neutral_sentence": neutral_sentence,
            "instruction_components": {
                "semantic_tags": [],
                "structural_metrics": {},
                "signature_rules": [],
            },
            "thinking_process": thinking_process,
            "output": output,
        }
        self.items[output] = item

        return True

    def save_character_semantic_tags(self, character: str, semantic_tags: list[str]) -> None:
        saved = False
        for item in self.items.values():
            if item["character"] == character:
                item["instruction_components"]["semantic_tags"] = semantic_tags
                saved = True
        if not saved:
            raise ValueError(f"角色 {character} 未找到")

    def save_character_signature_rules(self, character: str, signature_rules: list[str]) -> None:
        saved = False
        for item in self.items.values():
            if item["character"] == character:
                item["instruction_components"]["signature_rules"] = signature_rules
                saved = True
        if not saved:
            raise ValueError(f"角色 {character} 未找到")

    def save_component(
        self,
        output: str,
        structural_metrics: Optional[dict[str, float]] = None,
        semantic_tags: Optional[list[str]] = None,
        signature_rules: Optional[list[str]] = None,
        # 兼容 v1 参数名
        syntactic_vector: Optional[dict[str, float]] = None,
        pragmatic_styles: Optional[list[str]] = None,
    ) -> None:
        if output not in self.items:
            raise ValueError(f"风格句: {output} 似乎未加载")

        if structural_metrics is None and syntactic_vector is not None:
            structural_metrics = syntactic_vector
        if semantic_tags is None and pragmatic_styles is not None:
            semantic_tags = pragmatic_styles

        comp = self.items[output]["instruction_components"]
        if structural_metrics is not None:
            comp["structural_metrics"] = structural_metrics
        if semantic_tags is not None:
            comp["semantic_tags"] = semantic_tags
        if signature_rules is not None:
            comp["signature_rules"] = signature_rules

    @staticmethod
    def _verify_validity(item: DatasetItem) -> bool:
        if not all((
            item["character"],
            item["neutral_sentence"],
            item["output"],
            item["instruction_components"],
        )):
            return False

        comp = item["instruction_components"]
        dense = comp.get("structural_metrics", {})

        has_dense = all(key in dense for key in DENSE_METRIC_KEYS)
        has_semantic = len(comp.get("semantic_tags", [])) > 0
        has_signature = len(comp.get("signature_rules", [])) > 0

        return has_dense and has_semantic and has_signature

    @staticmethod
    def _oversampling(items_list: list[DatasetItem], item: DatasetItem) -> None:
        """
        过采样部分学习难度大的语义标签。

        1. `tsundere` `sharp_tongued` `proud` * 5
        2. `tsukkomi` `chuunibyou` `yandere` `airhead` * 3
        """
        difficult_labels_5x = {"tsundere", "sharp_tongued", "proud"}
        difficult_labels_3x = {"tsukkomi", "chuunibyou", "yandere", "airhead"}

        semantic_tags = set(item["instruction_components"].get("semantic_tags", []))

        if semantic_tags & difficult_labels_5x:
            for _ in range(5):
                items_list.append(item)
        elif semantic_tags & difficult_labels_3x:
            for _ in range(3):
                items_list.append(item)

    def output(self, output_path: Path, oversampling: bool = False) -> None:
        items = list(self.items.values())
        valid_items: list[DatasetItem] = []
        valid_items_count = 0

        for item in items:
            if not self._verify_validity(item):
                continue
            valid_items.append(item)
            valid_items_count += 1

            if oversampling:
                self._oversampling(valid_items, item)

        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(valid_items, f, ensure_ascii=False, indent=2)

        print(
            f"训练集已导出至 {output_path}, "
            f"总有效训练集数量: {len(valid_items)}, "
            f"跳过数量: {len(items) - valid_items_count}"
        )

dataset_storage = DatasetStorage()

### Load Neutral Sentences

In [27]:
neutral_sentences_jsonl_file = Path("./data/neutral_sentences_with_CoT.jsonl")

EN_NAME_TO_ZH = {"Muice": "沐雪",
                 "Ayaka": "神里绫华",
                 "Zhongli": "钟离",
                 "Hutao": "胡桃",
                 "Haruhi": "凉宫春日",
                 "Liyunlong": "李云龙",
                 "Sheldon": "谢尔顿",
                 "Wukong": "孙悟空",
                 }

class NSFileItem(TypedDict):
    character: str
    original: str
    neutral: str
    CoT: str

def load_jsonl_file(jsonl_file: Path) -> int:
    with open(jsonl_file, "r", encoding="utf-8") as f:
        jsonl_file_lines = f.readlines()

    duplicated_count = 0

    for line in tqdm(jsonl_file_lines, desc=f"Loading {jsonl_file.name}"):
    # for line in jsonl_file_lines:
        if line := line.rstrip():
            item: NSFileItem = json.loads(line)
            character = EN_NAME_TO_ZH.get(item["character"], item["character"])
            neutral = item["neutral"]
            success = dataset_storage.new_item(character, neutral, item['CoT'], item["original"])
            if not success:
                duplicated_count += 1

    return duplicated_count

duplicated_count = load_jsonl_file(neutral_sentences_jsonl_file)
avg_similarity = dataset_storage.calc_avg_similarity()
print(f"成功加载了 {len(dataset_storage)} 条训练集条目, 重复条目数量: {duplicated_count}, 平均相似度: {avg_similarity:.4f}")

Loading neutral_sentences_with_CoT.jsonl:   0%|          | 0/10921 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ../Models/bge-large-zh-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Calculating average similarity: 100%|██████████| 9813/9813 [00:00<00:00, 12294.70it/s]

成功加载了 9813 条训练集条目, 重复条目数量: 1108, 平均相似度: 0.5364


### Process Signature Layer Vectors

In [28]:
def get_lexical_keywords_from_file(file: Path, top_n: int = 25) -> list[str]:
    with open(file, "r", encoding="utf-8") as f:
        data: dict[str, float] = json.loads(f.read())
    return list(data.keys())[:top_n]

def load_punctuation_signature(file: Path) -> dict[str, list[str]]:
    rows = json.loads(file.read_text(encoding="utf-8"))
    return {row["role"]: row.get("S_signature", []) for row in rows}

def build_signature_rules(lexical_keywords: list[str], punctuation_symbols: list[str]) -> list[str]:
    rules: list[str] = []
    if lexical_keywords:
        rules.append("Lexical cues: " + ", ".join(lexical_keywords[:25]))
    if punctuation_symbols:
        rules.append("Punctuation tendencies: " + ", ".join(punctuation_symbols[:10]))
    return rules

LEXICAL_DIR = Path("./outputs/style/signature/lexical")
PUNCTUATION_SIGNATURE_FILE = Path("./outputs/style/signature/punctuation/symbol_signature.json")

CHARACTER_TO_FILE_STEM = {
    "沐雪": "muice",
    "神里绫华": "ayaka",
    "钟离": "zhongli",
    "胡桃": "hutao",
    "凉宫春日": "haruhi",
    "李云龙": "liyunlong",
    "谢尔顿": "sheldon",
    "孙悟空": "wukong"
}

punctuation_signature_map = load_punctuation_signature(PUNCTUATION_SIGNATURE_FILE)

for character, file_stem in CHARACTER_TO_FILE_STEM.items():
    lexical_file = LEXICAL_DIR / f"{file_stem}_pmi_filtered.json"
    lexical_keywords = get_lexical_keywords_from_file(lexical_file, top_n=25)
    punctuation_symbols = punctuation_signature_map.get(character, [])

    signature_rules = build_signature_rules(lexical_keywords, punctuation_symbols)
    dataset_storage.save_character_signature_rules(character, signature_rules)

print("成功注入 Signature 层（Lexical + Punctuation）规则")

成功注入 Signature 层（Lexical + Punctuation）规则


### Process Dense Structural Metrics (v_dense)

#### Load Dense Metrics from Normalized Style Vectors

In [29]:
DENSE_STYLE_FILE = Path("./outputs/style/dense/normalized/style_vector_phase2_minmax.json")

def _to_float(value: object, default: float = 0.0) -> float:
    try:
        return float(value)  # type: ignore[arg-type]
    except (TypeError, ValueError):
        return default

def load_dense_metrics_by_role(file_path: Path) -> dict[str, dict[str, float]]:
    rows = json.loads(file_path.read_text(encoding="utf-8"))
    role_to_metrics: dict[str, dict[str, float]] = {}

    for row in rows:
        role = row.get("role")
        if not role:
            continue

        role_to_metrics[role] = {
            "mdd": _to_float(row.get("mdd", row.get("MDD", 0.0))),
            "verb_noun_ratio": _to_float(row.get("verb_noun_ratio", 0.0)),
            "question_ratio": _to_float(row.get("question_ratio", 0.0)),
            "imperative_ratio": _to_float(row.get("imperative_ratio", 0.0)),
            "fragment_ratio": _to_float(row.get("fragment_ratio", 0.0)),
            "marker_density": _to_float(row.get("marker_density", 0.0)),
        }

    return role_to_metrics

dense_role_metrics = load_dense_metrics_by_role(DENSE_STYLE_FILE)
print(f"成功加载 {len(dense_role_metrics)} 个角色的稠密结构向量")

成功加载 9 个角色的稠密结构向量


#### Inject Dense Metrics to All Character Samples

In [30]:
TARGET_CHARACTERS = ["沐雪", "神里绫华", "钟离", "胡桃", "凉宫春日", "李云龙", "谢尔顿", "孙悟空"]

for character in TARGET_CHARACTERS:
    if character not in dense_role_metrics:
        raise ValueError(f"稠密向量缺失角色: {character}")

    metrics = dense_role_metrics[character]
    updated = 0
    for item in dataset_storage.items.values():
        if item["character"] == character:
            dataset_storage.save_component(item["output"], structural_metrics=metrics)
            updated += 1

    print(f"角色 {character} 已注入 6 维 dense 结构向量，样本数: {updated}")

print("成功注入 Dense 层结构向量")

角色 沐雪 已注入 6 维 dense 结构向量，样本数: 2462
角色 神里绫华 已注入 6 维 dense 结构向量，样本数: 1125
角色 钟离 已注入 6 维 dense 结构向量，样本数: 378
角色 胡桃 已注入 6 维 dense 结构向量，样本数: 751
角色 凉宫春日 已注入 6 维 dense 结构向量，样本数: 997
角色 李云龙 已注入 6 维 dense 结构向量，样本数: 1746
角色 谢尔顿 已注入 6 维 dense 结构向量，样本数: 1656
角色 孙悟空 已注入 6 维 dense 结构向量，样本数: 698
成功注入 Dense 层结构向量


### Process Semantic Style Layer Vectors

#### Read Semantic Tag Files for Each Corpus

In [31]:
PRAGMATIC_DIR = Path("./outputs/style/semantic/prag_vectors")

CHARACTER_TO_PRAG_FILE = {
    "沐雪": "muice.jsonl",
    "神里绫华": "ayaka.jsonl",
    "钟离": "zhongli.jsonl",
    "胡桃": "hutao.jsonl",
    "凉宫春日": "haruhi.jsonl",
    "李云龙": "liyunlong.jsonl",
    "谢尔顿": "sheldon.jsonl",
    "孙悟空": "wukong.jsonl"
}

class RawPragmaticItem(TypedDict):
    prompt: str
    response: str
    pragmatic_styles: list[dict[str, float]]

class SemanticItem(TypedDict):
    response: str
    semantic_tags: list[str]

def read_semantic_jsonl_file(jsonl_file: Path, threshold: float = 0.4) -> list[SemanticItem]:
    items: list[SemanticItem] = []
    with open(jsonl_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            raw_item: RawPragmaticItem = json.loads(line)
            score_map: dict[str, float] = {}
            for vec in raw_item.get("pragmatic_styles", []):
                score_map.update(vec)

            semantic_tags = [
                key for key, value in score_map.items()
                if float(value) > threshold
            ]
            items.append({"response": raw_item["response"], "semantic_tags": semantic_tags})

    return items

updated = 0
skipped = 0

for _, file_name in CHARACTER_TO_PRAG_FILE.items():
    semantic_items = read_semantic_jsonl_file(PRAGMATIC_DIR / file_name, threshold=0.4)
    for item in semantic_items:
        try:
            dataset_storage.save_component(item["response"], semantic_tags=item["semantic_tags"])
            updated += 1
        except ValueError:
            skipped += 1

print(f"更新了 {updated} 条语义标签，跳过 {skipped} 条未匹配 response")

更新了 9983 条语义标签，跳过 1762 条未匹配 response


### Export Training Set File

In [32]:
OUTPUT_PATH = Path("./data/llm_train.json")
OVERSAMPLING_OUTPUT_PATH = Path("./data/llm_train_oversampling.json")

# 对部分角色风格向量进行修正
for item in dataset_storage.items.values():
    if item["character"] == "沐雪":
        item["instruction_components"]["signature_rules"][1] = "Punctuation tendencies: ..., ⭐, ?!, multi-!, ♪, ❤"
    if item["character"] == "孙悟空":
        item["instruction_components"]["semantic_tags"] = ["energetic", "playful", "irritable", "loyal", "confident"]

dataset_storage.output(OUTPUT_PATH)
dataset_storage.output(OVERSAMPLING_OUTPUT_PATH, oversampling=True)

训练集已导出至 data\llm_train.json, 总有效训练集数量: 8272, 跳过数量: 1541
训练集已导出至 data\llm_train_oversampling.json, 总有效训练集数量: 9521, 跳过数量: 1541


## Define Model Structure & Data Loader

In [ ]:
import json
from pathlib import Path
from typing_extensions import TypedDict

import torch
import torch.nn as nn
from peft import LoraConfig, get_peft_model
from torch.utils.data import Dataset
from transformers import AutoTokenizer, Qwen3ForCausalLM
from transformers.data.data_collator import DataCollatorWithPadding

class InstructionComponents(TypedDict):
    semantic_tags: list[str]
    structural_metrics: dict[str, float]
    signature_rules: list[str]

class DatasetItem(TypedDict):
    character: str
    neutral_sentence: str
    instruction_components: InstructionComponents
    thinking_process: str
    output: str

class StyleDataset(Dataset):
    def __init__(
        self,
        path: Path,
        tokenizer,
        max_length: int = 512,
    ):
        self.tokenizer = tokenizer

        # 过滤超长样本而不是截断
        self.data: list[DatasetItem] = []
        raw_data = json.loads(path.read_text(encoding="utf-8"))
        dropped = 0
        for item in raw_data:
            # 快速估算 token 长度
            approx_len = self._estimate_token_length(item, tokenizer)
            if approx_len <= max_length:
                self.data.append(item)
            else:
                dropped += 1

        if dropped > 0:
            print(f"Dropped {dropped}/{len(raw_data)} samples "
                f"exceeding max_length={max_length}")

        self.syntactic_dims = [
            "mdd", "verb_noun_ratio", "question_ratio",
            "imperative_ratio", "fragment_ratio", "marker_density"
        ]
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def _estimate_token_length(self, item: DatasetItem, tokenizer) -> int:
        system_prompt = (
            "You are an expert in stylistic rewriting. "
            "Your task is to rewrite the neutral sentence into the target character's specific pragmatics and formatting style."
        )
        user_prompt = (
            f"Target Character: {item['character']}\n"
            f"Semantic Traits: {', '.join(item['instruction_components'].get('semantic_tags', []))}\n"
            f"Formatting Signature: {' '.join(item['instruction_components'].get('signature_rules', []))}\n"
            f"Neutral Content: {item['neutral_sentence']}\n"
        )
        assistant_response = (item.get("thinking_process") or "").strip() + "\n\n" + (item.get("output") or "").strip()

        full_text = self.tokenizer.apply_chat_template(
            [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
                {"role": "assistant", "content": assistant_response},
            ],
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=False,
        )
        return len(tokenizer(full_text, truncation=False, padding=False)["input_ids"])

    def __getitem__(self, idx: int):
        item = self.data[idx]
        instr = item["instruction_components"]

        # (1) 提取 6 维连续结构特征
        syntactic_vec = torch.tensor(
            [instr.get("structural_metrics", {}).get(dim, 0.0) for dim in self.syntactic_dims],
            dtype=torch.float32
        )

        # (2) 将离散语义与签名规则注入文本通道
        semantic_tags = ", ".join(instr.get("semantic_tags", [])) or "None"
        signature_rules = " ".join(instr.get("signature_rules", [])) or "None"

        system_prompt = (
            "You are an expert in stylistic rewriting. "
            "Your task is to rewrite the neutral sentence into the target character's specific pragmatics and formatting style."
        )
        user_prompt = (
            f"Target Character: {item['character']}\n"
            f"Semantic Traits: {semantic_tags}\n"
            f"Formatting Signature: {signature_rules}\n"
            f"Neutral Content: {item['neutral_sentence']}\n"
        )

        cot_text = (item.get("thinking_process") or "").strip()
        output_text = (item.get("output") or "").strip()
        if cot_text:
            cot_prefix_text = f"{cot_text}\n\n"
            assistant_response = cot_prefix_text + output_text
        else:
            cot_prefix_text = ""
            assistant_response = output_text

        # (3) Tokenize 完整文本和 Prompt（先关闭截断）
        full_text = self.tokenizer.apply_chat_template(
            [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
                {"role": "assistant", "content": assistant_response},
            ],
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=False,
        )

        prompt_only_text = self.tokenizer.apply_chat_template(
            [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )

        # 获取无截断的完整 IDs
        full_ids = self.tokenizer(full_text, truncation=False, padding=False)["input_ids"]
        prompt_ids = self.tokenizer(prompt_only_text, truncation=False, padding=False)["input_ids"]
        prompt_len = len(prompt_ids)

        # (4) 在完整长度下构建 Labels
        labels = full_ids.copy()
        effective_prompt_len = min(prompt_len, len(labels))
        labels[:effective_prompt_len] = [-100] * effective_prompt_len

        # (5) 构建 CoT / 输出 token 掩码（用于加权 LM loss）
        cot_mask = [0] * len(full_ids)
        output_mask = [0] * len(full_ids)

        assistant_start = min(prompt_len, len(full_ids))
        cot_token_len = 0
        if cot_prefix_text:
            cot_token_len = len(
                self.tokenizer(
                    cot_prefix_text,
                    add_special_tokens=False,
                    truncation=False,
                    padding=False,
                )["input_ids"]
            )
        cot_end = min(assistant_start + cot_token_len, len(full_ids))

        for i in range(assistant_start, len(full_ids)):
            if labels[i] == -100:
                continue
            if i < cot_end:
                cot_mask[i] = 1
            else:
                output_mask[i] = 1

        # (6) 采用右截断，保留前缀和指令
        if len(full_ids) > self.max_length:
            input_ids = full_ids[:self.max_length]
            labels = labels[:self.max_length]
            cot_mask = cot_mask[:self.max_length]
            output_mask = output_mask[:self.max_length]
        else:
            input_ids = full_ids

        return {
            "input_ids": input_ids,
            "syntactic_vec": syntactic_vec,
            "labels": labels,
            "cot_mask": cot_mask,
            "output_mask": output_mask,
        }

class StyleDataCollator(DataCollatorWithPadding):
    """动态 padding + 6 维结构向量拼接 + 手动处理 labels 和 token masks"""

    def __init__(self, tokenizer, **kwargs):
        super().__init__(tokenizer=tokenizer, padding=True, **kwargs)

    def __call__(self, features):
        syntactic_vecs = torch.stack([f["syntactic_vec"] for f in features])
        labels = [f["labels"] for f in features]
        cot_masks = [f["cot_mask"] for f in features]
        output_masks = [f["output_mask"] for f in features]

        base_features = [
            {k: v for k, v in f.items() if k not in ("syntactic_vec", "labels", "cot_mask", "output_mask")}
            for f in features
        ]
        batch = super().__call__(base_features)

        max_length = batch["input_ids"].shape[1]

        def _pad_sequences(sequences: list[list[int]], pad_value: int) -> list[list[int]]:
            padded: list[list[int]] = []
            for seq in sequences:
                truncated_seq = seq[:max_length]
                padding_length = max_length - len(truncated_seq)
                padded.append(truncated_seq + [pad_value] * padding_length)
            return padded

        batch["labels"] = torch.tensor(_pad_sequences(labels, -100), dtype=torch.long)
        batch["cot_mask"] = torch.tensor(_pad_sequences(cot_masks, 0), dtype=torch.float32)
        batch["output_mask"] = torch.tensor(_pad_sequences(output_masks, 0), dtype=torch.float32)
        batch["syntactic_vec"] = syntactic_vecs
        return batch

class StyleEncoder(nn.Module):
    def __init__(self, input_dim, out_dim):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(input_dim, out_dim // 2),
            nn.ReLU(),
            nn.Linear(out_dim // 2, out_dim),
            nn.Tanh(),
        )

    def forward(self, syntactic_vec):
        return self.proj(syntactic_vec)

class StyleConditionedLoRAModel(nn.Module):
    def __init__(self, model_name_or_path: str | Path, syntactic_dim: int = 6, lora_r=16, lora_alpha=16, max_seq_length=2048):
        super().__init__()

        tokenizer = AutoTokenizer.from_pretrained(str(model_name_or_path), use_fast=True)
        tokenizer.model_max_length = max_seq_length
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_side = "right"

        base_model = Qwen3ForCausalLM.from_pretrained(
            str(model_name_or_path),
            torch_dtype=torch.bfloat16,
            attn_implementation="sdpa",
            use_cache=False,
        )
        
        base_model.gradient_checkpointing_enable()
        base_model.enable_input_require_grads()

        lora_config = LoraConfig(
            r=lora_r,
            lora_alpha=lora_alpha,
            target_modules=[
                "q_proj", "k_proj", "v_proj", "o_proj",
                "gate_proj", "up_proj", "down_proj",
            ],
            lora_dropout=0.0,
            bias="none",
            task_type="CAUSAL_LM",
        )

        self.peft_model = get_peft_model(base_model, lora_config)
        self.tokenizer = tokenizer

        self.hidden_size = int(getattr(self.peft_model.config, "hidden_size"))
        self.style_encoder = StyleEncoder(syntactic_dim, self.hidden_size)

        calc_dtype = torch.bfloat16
        self.style_encoder.to(calc_dtype)
        if torch.cuda.is_available():
            self.style_encoder.to(torch.cuda.current_device())

    def get_input_embeddings(self):
        return self.peft_model.get_input_embeddings()

    def forward(self, input_ids, attention_mask, labels, syntactic_vec):
        _ = labels

        style_dtype = next(self.style_encoder.parameters()).dtype
        style_emb = self.style_encoder(syntactic_vec.to(style_dtype))
        style_emb_prefix = style_emb.unsqueeze(1)

        token_embeds = self.peft_model.get_input_embeddings().forward(input_ids)
        inputs_embeds = torch.cat([style_emb_prefix.to(token_embeds.dtype), token_embeds], dim=1)

        prefix_mask = torch.ones(
            attention_mask.shape[0], 1,
            dtype=torch.long,
            device=attention_mask.device,
        )
        new_attention_mask = torch.cat([prefix_mask, attention_mask], dim=1)

        outputs = self.peft_model(
            inputs_embeds=inputs_embeds,
            attention_mask=new_attention_mask,
            output_hidden_states=True,
        )

        last_hidden = outputs.hidden_states[-1]
        return {
            "logits": outputs.logits[:, :-1, :],
            "token_hidden_states": last_hidden[:, 1:, :],
        }

/root/miniconda3/lib/python3.12/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.8.0+cu128).


## Load Model & Training Data

In [ ]:
from transformers.optimization import get_scheduler
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
from pathlib import Path

MODEL_PATH = "/root/autodl-tmp/Qwen3-1.7B/"
DATASET_PATH = Path("/root/OtakuLab/data/llm_train_oversampling.json")
# MODEL_PATH = "../Models/Qwen3-1.7B/"
# DATASET_PATH = Path("./data/llm_train_oversampling.json")
TEST_SIZE = 0.2

BATCH_SIZE = 8
LEARNING_RATE = 3e-5
WARMUP_STEP = 150
EPOCH = 2

syntactic_dims = [
    "mdd", "verb_noun_ratio", "question_ratio",
    "imperative_ratio", "fragment_ratio", "marker_density",
]
syntactic_dim_length = len(syntactic_dims)

print("Preparing for train...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

styled_model = StyleConditionedLoRAModel(
    MODEL_PATH,
    syntactic_dim=syntactic_dim_length,
    lora_r=16,
    lora_alpha=16,
)

tokenizer = styled_model.tokenizer
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print(f"Set tokenizer.pad_token to {tokenizer.eos_token}")

# 强制设置右侧填充，确保与 labels 的 padding 逻辑严格对齐
tokenizer.padding_side = "right"

print("Loading Dataset...")
train_dataset = StyleDataset(DATASET_PATH, tokenizer)
val_dataset = StyleDataset(DATASET_PATH, tokenizer)

indices = list(range(len(train_dataset)))
train_indices, val_indices = train_test_split(indices, test_size=TEST_SIZE, random_state=42)

train_data = Subset(train_dataset, train_indices)
val_data = Subset(val_dataset, val_indices)

collator = StyleDataCollator(tokenizer, pad_to_multiple_of=8)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator, persistent_workers=False)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collator)
print(f"Successfully loaded dataset, length: {len(train_dataset)}")

sample = train_data[0]
labels = sample["labels"]  # type:ignore
input_ids = sample["input_ids"]  # type:ignore

# 找出第一个不是 -100 的索引
start_idx = 0
while start_idx < len(labels) and labels[start_idx] == -100:
    start_idx += 1

print("\n--- Sanity Check ---")
print("Prompt (Masked):", tokenizer.decode(input_ids[:start_idx]))
print("Output (Trained):", tokenizer.decode(input_ids[start_idx:]))
print("--------------------\n")

optimizer = torch.optim.AdamW(styled_model.parameters(), lr=LEARNING_RATE)

num_training_steps = len(train_loader) * EPOCH
lr_scheduler = get_scheduler(
    "cosine",
    optimizer=optimizer,
    num_warmup_steps=WARMUP_STEP,
    num_training_steps=num_training_steps,
)

Preparing for train...


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

Loading Dataset...
Dropped 14/9521 samples exceeding max_length=512
Dropped 14/9521 samples exceeding max_length=512
Successfully loaded dataset, length: 9507

--- Sanity Check ---
Prompt (Masked): <|im_start|>system
You are an expert in stylistic rewriting. Your task is to rewrite the neutral sentence into the target character's specific pragmatics and formatting style.<|im_end|>
<|im_start|>user
Target Character: 沐雪
Semantic Traits: defensive, irritable
Formatting Signature: Lexical cues: 喵, 沐沐, AI, ~, 沐雪, 恼, 女孩子, ⭐, 聊天, 唔, 呀, 可爱, 答, 不行, 笨蛋, 雪雪, 直播, 脸红, 睡觉, www, 谢谢, 变态, 骂, 嘛, 杂鱼 Punctuation tendencies: ..., ⭐, ?!, multi-!, ♪, ❤
Neutral Content: 你说谁是傻孩子？其实你才是。<|im_end|>
<|im_start|>assistant
<think>
她依
Output (Trained): 防御与易怒语义选用反问反击，高疑问比与碎片率造就急促节奏，最后借“恼”标签、重复词及多感叹号强化傲娇情绪。
</think>

说谁是傻孩子呢？你才是傻孩子！（恼）<|im_end|>

--------------------



## Train Model

In [3]:
from tqdm import tqdm
import math

VAL_INTERVAL = 250
COT_TOKEN_WEIGHT = 0.7
OUTPUT_TOKEN_WEIGHT = 1.0

ce_loss_fn = nn.CrossEntropyLoss(ignore_index=-100, reduction="none")

def compute_weighted_lm_loss(
    logits: torch.Tensor,
    labels: torch.Tensor,
    cot_mask: torch.Tensor,
    output_mask: torch.Tensor,
    cot_weight: float,
    output_weight: float,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    token_loss = ce_loss_fn(
        logits.reshape(-1, logits.shape[-1]),
        labels.reshape(-1),
    ).view_as(labels)

    valid_mask = (labels != -100).float()
    cot_token_mask = cot_mask * valid_mask
    output_token_mask = output_mask * valid_mask

    cot_denom = cot_token_mask.sum().clamp(min=1.0)
    output_denom = output_token_mask.sum().clamp(min=1.0)

    L_cot = (token_loss * cot_token_mask).sum() / cot_denom
    L_final_output = (token_loss * output_token_mask).sum() / output_denom

    token_weights = cot_token_mask * cot_weight + output_token_mask * output_weight
    denom = token_weights.sum().clamp(min=1.0)
    L_lm = (token_loss * token_weights).sum() / denom

    return L_lm, L_cot, L_final_output


styled_model.train()

global_step = 0

for epoch in range(EPOCH):
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EPOCH}")

    for batch in progress_bar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        syntactic_vec = batch["syntactic_vec"].to(device)
        cot_mask = batch["cot_mask"].to(device)
        output_mask = batch["output_mask"].to(device)

        outputs = styled_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            syntactic_vec=syntactic_vec,
        )

        L_lm, L_cot, L_final_output = compute_weighted_lm_loss(
            outputs["logits"],
            labels,
            cot_mask,
            output_mask,
            COT_TOKEN_WEIGHT,
            OUTPUT_TOKEN_WEIGHT,
        )

        L_lm.backward()
        torch.nn.utils.clip_grad_norm_(styled_model.parameters(), max_norm=1.0)
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

        global_step += 1

        progress_bar.set_postfix({
            "L_lm": L_lm.item(),
            "L_cot": L_cot.item(),
            "L_out": L_final_output.item(),
            "LR": lr_scheduler.get_last_lr()[0],
        })

        if global_step % VAL_INTERVAL == 0:
            styled_model.eval()

            val_loss_lm = 0.0
            val_loss_cot = 0.0
            val_loss_output = 0.0
            val_steps = 0

            with torch.no_grad():
                for val_batch in val_loader:
                    val_input_ids = val_batch["input_ids"].to(device)
                    val_attention_mask = val_batch["attention_mask"].to(device)
                    val_labels = val_batch["labels"].to(device)
                    val_syntactic_vec = val_batch["syntactic_vec"].to(device)
                    val_cot_mask = val_batch["cot_mask"].to(device)
                    val_output_mask = val_batch["output_mask"].to(device)

                    val_outputs = styled_model(
                        input_ids=val_input_ids,
                        attention_mask=val_attention_mask,
                        labels=val_labels,
                        syntactic_vec=val_syntactic_vec,
                    )

                    L_lm_val, L_cot_val, L_output_val = compute_weighted_lm_loss(
                        val_outputs["logits"],
                        val_labels,
                        val_cot_mask,
                        val_output_mask,
                        COT_TOKEN_WEIGHT,
                        OUTPUT_TOKEN_WEIGHT,
                    )

                    val_loss_lm += L_lm_val.item()
                    val_loss_cot += L_cot_val.item()
                    val_loss_output += L_output_val.item()
                    val_steps += 1

            avg_val_lm = val_loss_lm / max(1, val_steps)
            avg_val_cot = val_loss_cot / max(1, val_steps)
            avg_val_output = val_loss_output / max(1, val_steps)
            ppl = math.exp(min(avg_val_lm, 20))

            print(
                f"Step {global_step}: "
                f"Train L_lm={L_lm.item():.4f}, "
                f"Train L_cot={L_cot.item():.4f}, "
                f"Train L_out={L_final_output.item():.4f}"
            )
            print(
                f"Step {global_step}: "
                f"Val PPL={ppl:.4f}, "
                f"Val L_lm={avg_val_lm:.4f}, "
                f"Val L_cot={avg_val_cot:.4f}, "
                f"Val L_out={avg_val_output:.4f}"
            )

            styled_model.train()

Prompt Dropout initial state: format_only=0.000, pragmatic_only=0.000, both=0.000, none=1.000


Epoch 1/2:   0%|          | 0/633 [00:00<?, ?it/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
Epoch 1/2:  16%|█▌        | 100/633 [02:22<2:08:51, 14.51s/it, L_total=2.26, L_lm=2.26, L_cot=2.99, L_out=1.44, L_style=1.16, P_drop=0, LR=2.67e-5]

Step 100: Train L_total=2.2587, Train L_lm=2.2587, Train L_cot=2.9853, Train L_out=1.4357, Train L_style=1.1611, PromptDrop=0.000
Step 100: Val PPL=11.1305, Val L_lm=2.4097, Val L_cot=3.1391, Val L_out=1.6405, Val L_style=1.2639


Epoch 1/2:  32%|███▏      | 200/633 [04:43<1:44:28, 14.48s/it, L_total=1.53, L_lm=1.53, L_cot=1.72, L_out=1.18, L_style=1.55, P_drop=0, LR=3.98e-5]

...

Epoch 2/2:  74%|███████▍  | 467/633 [11:15<40:15, 14.55s/it, L_total=1.35, L_lm=1.35, L_cot=1.12, L_out=1.53, L_style=1.28, P_drop=0, LR=2.14e-6]    

Step 1100: Train L_total=1.3491, Train L_lm=1.3491, Train L_cot=1.1199, Train L_out=1.5270, Train L_style=1.2796, PromptDrop=0.000
Step 1100: Val PPL=3.4630, Val L_lm=1.2421, Val L_cot=1.1831, Val L_out=1.2994, Val L_style=1.4670


Epoch 2/2:  90%|████████▉ | 567/633 [13:38<15:59, 14.53s/it, L_total=1.31, L_lm=1.31, L_cot=1.15, L_out=1.45, L_style=1.45, P_drop=0, LR=3.44e-7]    

Step 1200: Train L_total=1.3118, Train L_lm=1.3118, Train L_cot=1.1504, Train L_out=1.4466, Train L_style=1.4545, PromptDrop=0.000
Step 1200: Val PPL=3.4591, Val L_lm=1.2410, Val L_cot=1.1814, Val L_out=1.2989, Val L_style=1.4593


Epoch 2/2: 100%|██████████| 633/633 [14:41<00:00,  1.39s/it, L_total=1.16, L_lm=1.16, L_cot=1.14, L_out=1.19, L_style=1.8, P_drop=0, LR=0]           


## Export Model

In [4]:
ver = "4.3"

SAVE_DIR = Path(f"/root/OtakuLab/outputs/styled-qwen-v{ver}-{LEARNING_RATE}")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Saving model to {SAVE_DIR}...")

# 1️⃣ 保存 LoRA 权重（仅包含可训练部分）
styled_model.peft_model.save_pretrained(SAVE_DIR / "lora")  # type:ignore

# 2️⃣ 保存 StyleEncoder 权重
torch.save(styled_model.style_encoder.state_dict(), SAVE_DIR / "style_encoder.pt")

# 4️⃣ 保存 Tokenizer 配置
tokenizer.save_pretrained(SAVE_DIR / "tokenizer")

print("✅ Model checkpoint saved successfully.")


Saving model to /root/OtakuLab/outputs/styled-qwen-v4.3-4e-05...
✅ Model checkpoint saved successfully.


## Load Model

Before running this cell, ensure that all code in the `Define Model Structure & Data Loader` cell has been executed and model training is complete.

In [5]:
from pathlib import Path

import torch
from peft import PeftModel
from transformers import AutoTokenizer, Qwen3ForCausalLM

MODEL_PATH = Path("/root/autodl-tmp/Qwen3-1.7B/")
SAVE_DIR = Path("./outputs/styled-qwen-v5.0-3e-05")
# DPO_SAVE_DIR: Path | None = Path("./outputs/model/styled-qwen-dpo-v1/checkpoint-150")
DPO_SAVE_DIR: Path | None = None

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_ID = DPO_SAVE_DIR or SAVE_DIR / "lora"

syntactic_dims = [
    "mdd", "verb_noun_ratio", "question_ratio",
    "imperative_ratio", "fragment_ratio", "marker_density",
]

tokenizer_path = SAVE_DIR / "tokenizer"
tokenizer = AutoTokenizer.from_pretrained(
    str(tokenizer_path if tokenizer_path.exists() else MODEL_PATH),
    use_fast=True,
    padding_side="right",
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = Qwen3ForCausalLM.from_pretrained(
    str(MODEL_PATH),
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",
)
model = PeftModel.from_pretrained(base_model, str(MODEL_ID))
model = model.to(DEVICE)
hidden_size = int(getattr(model.config, "hidden_size"))

# === 加载 StyleEncoder ===
style_encoder = StyleEncoder(len(syntactic_dims), hidden_size)
style_encoder.load_state_dict(torch.load(SAVE_DIR / "style_encoder.pt", map_location=DEVICE))
style_encoder = style_encoder.to(device=DEVICE, dtype=torch.bfloat16)
style_encoder.eval()

model.eval()
print("Styled model loaded and ready for inference.")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

✅ Styled model loaded and ready for inference.


## Call Model for Inference

In [6]:
import torch

@torch.inference_mode()
def generate_styled_response(
    neutral_sentence: str,
    structural_metrics: dict[str, float],
    character_name: str = "Ayaka",
    semantic_tags: list[str] | None = None,
    signature_rules: list[str] | None = None,
    temperature: float = 0.5,
    top_p: float = 0.9,
    max_new_tokens: int = 768,
    repetition_penalty: float = 1.05,
    enable_thinking: bool = True,
    force_inject_chinese_anchor: bool = True,
 ):
    """输入中性句与 6 维结构向量，生成风格化响应"""

    semantic_tags = semantic_tags or []
    signature_rules = signature_rules or []

    semantic_text = ", ".join(semantic_tags) if semantic_tags else "None"
    signature_text = " ".join(signature_rules) if signature_rules else "None"

    system_prompt = (
        "You are an expert in stylistic rewriting. "
        "Your task is to rewrite the neutral sentence into the target character's specific pragmatics and formatting style."
    )
    user_prompt = (
        f"Target Character: {character_name}\n"
        f"Semantic Traits: {semantic_text}\n"
        f"Formatting Signature: {signature_text}\n"
        f"Neutral Content: {neutral_sentence}\n"
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enable_thinking,
    )
    
    if force_inject_chinese_anchor:
        input_text = input_text.rstrip() + " <think>思考: "

    tokenized = tokenizer(input_text, return_tensors="pt").to(DEVICE)
    input_ids = tokenized["input_ids"]
    attention_mask = tokenized["attention_mask"]

    syntactic_tensor = torch.tensor(
        [structural_metrics.get(dim, 0.0) for dim in syntactic_dims],
        dtype=torch.bfloat16,
        device=DEVICE,
    ).unsqueeze(0)

    style_emb = style_encoder(syntactic_tensor)
    style_prefix = style_emb.unsqueeze(1)

    token_embeds = model.get_input_embeddings()(input_ids)  # type: ignore[operator]
    inputs_embeds = torch.cat([style_prefix, token_embeds], dim=1)

    prefix_mask = torch.ones((1, 1), dtype=torch.long, device=DEVICE)
    new_attention_mask = torch.cat([prefix_mask, attention_mask], dim=1)

    outputs = model.generate(
        inputs_embeds=inputs_embeds,
        attention_mask=new_attention_mask,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        do_sample=True,
        repetition_penalty=repetition_penalty,
    )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return result if not force_inject_chinese_anchor else "<think>思考: " + result

In [8]:
# --- Example: Muice-like style ---
neutral_sentence = "身高和体重是个人隐私，不能随便告诉你。"
styled_sentence = generate_styled_response(
    character_name="沐雪",
    neutral_sentence=neutral_sentence,
    structural_metrics={
        "mdd": 0.2201,
        "verb_noun_ratio": 0.8084,
        "question_ratio": 0.8631,
        "imperative_ratio": 0.4648,
        "fragment_ratio": 0.7843,
        "marker_density": 0.4233,
    },
    semantic_tags=["cute", "playful", "energetic", "airhead", "humorous"],
    signature_rules=[
        "Lexical cues: 喵, 沐沐, AI, ~, 沐雪, 恼, 女孩子, ⭐, 聊天, 唔, 呀, 可爱, 答, 不行, 笨蛋, 雪雪, 直播, 脸红, 睡觉, www, 谢谢, 变态, 骂, 嘛, 杂鱼",
        "Punctuation tendencies: ..., ⭐, ?!, multi-!, ♪, ❤",
    ],
    force_inject_chinese_anchor=True,
 )
print(f"{neutral_sentence} -> {styled_sentence}")
print()

# --- Example: Sunwukong Classic Style ---
neutral_sentence = "师父，如果你问别人恐怕不会有好结果，只有问了他才有主意。"
styled_response_2 = generate_styled_response(
    character_name="孙悟空",
    neutral_sentence=neutral_sentence,
    structural_metrics={
        "mdd": 0.3997,
        "verb_noun_ratio": 0.5705,
        "question_ratio": 0.7424,
        "imperative_ratio": 0.3252,
        "fragment_ratio": 0.556,
        "marker_density": 0.1086,
    },
    semantic_tags=["energetic", "playful", "irritable", "loyal", "confident"],
    signature_rules=[
        "Lexical cues: 师父, 老孙, 孙, 老, 哩, 莫, 甚么, 妖精, 怪, 儿, 等, 呆子, 甚, 不曾, 罢, 菩萨, 教, 不知, 若是, 泼, 却, 棍, 如今, 这般, 云",
        "Punctuation tendencies: (None)",
    ],
    force_inject_chinese_anchor=True,
 )
print(f"{neutral_sentence} -> {styled_response_2}")
print()

# --- Example: Tsundere Style ---
neutral_sentence = "谢谢你帮我。"
styled_response_3 = generate_styled_response(
    character_name="Katsumi",
    neutral_sentence=neutral_sentence,
    structural_metrics={
        "mdd": 0.45,
        "verb_noun_ratio": 0.55,
        "question_ratio": 0.18,
        "imperative_ratio": 0.26,
        "fragment_ratio": 0.48,
        "marker_density": 0.29,
    },
    semantic_tags=["tsundere", "defensive"],
    signature_rules=[
        "Lexical cues: 哼, 才没有, 笨蛋, 不要, 讨厌, 随便, 误会, 反正, 真是的, 讨厌鬼, 哎呀, 别说了, 坏, 麻烦",
        "Punctuation tendencies: ?!, multi-!",
    ],
    force_inject_chinese_anchor=True,
 )
print(f"{neutral_sentence} -> {styled_response_3}")

neutral_sentence = "这只鱼看着就感觉好吃"
styled_response_3 = generate_styled_response(
    character_name="李云龙",
    neutral_sentence=neutral_sentence,
    structural_metrics={
        "mdd": 0.4897,
        "verb_noun_ratio": 0.5131,
        "question_ratio": 1.0,
        "imperative_ratio": 1.0,
        "fragment_ratio": 0.9085,
        "marker_density": 0.0229,
    },
    signature_rules=[
        "Lexical cues: 鬼子, 咱们, 中国人, 敌人, 不怕, 侵略者, 军人, 保卫, 兄弟们, 李云龙, 别, 狠狠, 政委, 部队, 小子, 打仗, 独立, 祖国, 中国, 国家, 日本, 屁, 牺牲, 教训, 楚",
        "Punctuation tendencies: ?!, ...",
    ],
    semantic_tags=["rational", "kind", "responsible", "energetic", "serious"],
    force_inject_chinese_anchor=True,
 )
print(f"{neutral_sentence} -> {styled_response_3}")

身高和体重是个人隐私，不能随便告诉你。 -> <think>思考: 作为可爱活泼的AI，用高碎片率和疑问感制造跳跃节奏；利用“嘛”等语气词及问号、感叹号强化俏皮与互动感。
</think>

身高体重是个人隐私嘛，不能随便告诉你

师父，如果你问别人恐怕不会有好结果，只有问了他才有主意。 -> <think>思考: 以自信顽皮语义定调，高疑问与碎片率促短句连发，借“老孙”等自称及省略号强化急切口吻。
</think>

师父，你若问别人，怕没有好结果；只问了他，才有主意。

谢谢你帮我。 -> <think>思考: 作为傲娇与防御语义的结合体，她会先用“哼”或“讨厌”等词强化情绪，再通过高碎片率和祈使感制造急促节奏，最后用感叹号和多感叹号强化语气。
</think>

哼，谢了你帮我。
这只鱼看着就感觉好吃 -> <think>思考: 语义“负责、热血”与高碎片率结合，将中性陈述转为粗犷反问；极高问句/祈使比及动词占比强化命令感，最后用“咱们”“屁”等词汇和"?! "标点注入李云龙特有的豪爽与火辣。
</think>

你看看这鱼，就感觉是吃着好吃的！
